In [2]:
# Cell 1: Environment Setup and Data Ingestion
%pip install kagglehub pandas numpy

import os
import kagglehub
import pandas as pd
import numpy as np

print("--- Step 1: Downloading dataset from Kaggle ---")
# Automatically download the latest version of the dataset
download_path = kagglehub.dataset_download("CooperUnion/anime-recommendations-database")
print(f"Dataset downloaded and stored at: {download_path}\n")

# Define paths for both CSV files
anime_raw_path = os.path.join(download_path, "anime.csv")
rating_raw_path = os.path.join(download_path, "rating.csv")

# Load datasets into Pandas DataFrames
df_anime = pd.read_csv(anime_raw_path)
df_rating = pd.read_csv(rating_raw_path)

# Show basic structural information (Checklist I)
print("--- Step 2: Checking Raw Data Dimensions ---")
print(f"Anime Catalog Shape: {df_anime.shape[0]} rows, {df_anime.shape[1]} columns")
print(f"User Ratings Shape: {df_rating.shape[0]} rows, {df_rating.shape[1]} columns")

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   ------------------- -------------------- 6.0/12.5 MB 19.9 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.5 MB 23.3 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.5 MB 23.3 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.5 MB 23.3 MB/s eta 0:00:01
   ---------------------------------------- 12.5/12.5 MB 11.2 MB/s  0:00:01
Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl (159 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   -----


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\HP ELITEBOOK 820 G4\OneDrive\Escritorio\bootcamp\PROYECTOS\Mod_2_Proyecto_II\Anime Analytics Dashboard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Step 1: Downloading dataset from Kaggle ---


100%|██████████| 25.0M/25.0M [00:01<00:00, 15.0MB/s]

Extracting files...


Dataset downloaded and stored at: C:\Users\HP ELITEBOOK 820 G4\.cache\kagglehub\datasets\CooperUnion\anime-recommendations-database\versions\1

--- Step 2: Checking Raw Data Dimensions ---
Anime Catalog Shape: 12294 rows, 7 columns
User Ratings Shape: 7813737 rows, 3 columns


In [3]:
# Cell 2: Data Auditing and Cleaning Strategy
print("--- Step 3: Detecting Null Values in Anime Metadata ---")
print(df_anime.isnull().sum())

# Business Logic: 'Unknown' values in episodes column act as hidden nulls. We fix them.
# Logica de Negocio: Valores 'Unknown' en episodios actuan como nulos ocultos. Los corregimos.
df_anime['episodes'] = df_anime['episodes'].replace('Unknown', np.nan)
df_anime['episodes'] = pd.to_numeric(df_anime['episodes'])

# Drop rows with missing values in critical business metrics (rating, type, episodes)
# Eliminamos filas con nulos en metricas criticas para asegurar el rigor matematico posterior
df_clean_anime = df_anime.dropna(subset=['rating', 'type', 'episodes']).copy()

print(f"\n--- Step 4: Dimensions After Data Cleaning ---")
print(f"Clean Anime Catalog Shape: {df_clean_anime.shape[0]} rows")

--- Step 3: Detecting Null Values in Anime Metadata ---
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

--- Step 4: Dimensions After Data Cleaning ---
Clean Anime Catalog Shape: 11876 rows


In [4]:
# Cell 3: Data Governance & Sampling Bias Audit
print("--- Step 5: Auditing User Rating Behavior (Sampling Bias) ---")

# Calculate how many ratings each user has submitted
user_activity = df_rating['user_id'].value_counts()

print("User Engagement Statistics (Descriptive):")
print(f"Average ratings per user (Media): {user_activity.mean():.2f}")
print(f"Median ratings per user (Mediana): {user_activity.median():.2f}")
print(f"Max ratings by a single user (Maximo): {user_activity.max()}")

# Corporate Alert: Is there a bias towards hardcore users?
# Alerta Corporativa: ¿Existe un sesgo hacia usuarios hiperactivos que opaque al publico general?
heavy_users_share = (user_activity[user_activity > 100].count() / len(user_activity)) * 100
print(f"\nPercentage of users with >100 ratings: {heavy_users_share:.2f}%")

--- Step 5: Auditing User Rating Behavior (Sampling Bias) ---
User Engagement Statistics (Descriptive):
Average ratings per user (Media): 106.29
Median ratings per user (Mediana): 57.00
Max ratings by a single user (Maximo): 10227

Percentage of users with >100 ratings: 33.57%


In [6]:
# Cell 4: Business Data Transformation & Export for Power BI
print("--- Step 6: Generating Strategic Metrics per Anime ---")

# 1. Group ratings to calculate true community metrics
anime_stats = df_rating.groupby('anime_id').agg(
    total_votes=('rating', 'count'),
    actual_community_rating=('rating', lambda x: x[x != -1].mean()) # Exclude -1 (watched but not rated)
).reset_index()

# 2. Merge stats back into our clean anime catalog
df_segmented = pd.merge(df_clean_anime, anime_stats, on='anime_id', how='inner')

# 3. Clean and isolate the Primary Genre for clean Power BI filtering
df_segmented['primary_genre'] = df_segmented['genre'].apply(lambda x: x.split(',')[0] if pd.notnull(x) else 'Unknown')

# 4. Strategic Business Segmentation based on size/members
def segment_market_size(members):
    if members > 100000: return '1. Mass Market (Blockbuster)'
    elif members > 10000: return '2. Established Core'
    else: return '3. Niche / Long-Tail'

df_segmented['market_segment'] = df_segmented['members'].apply(segment_market_size)

# 5. FIXED: Ensure the directory exists before saving!
# CORREGIDO: Aseguramos que la carpeta exista antes de guardar para evitar el OSError
output_dir = 'data/processed'
os.makedirs(output_dir, exist_ok=True) 

# 6. Export the golden dataset to Power BI folder
output_file = os.path.join(output_dir, 'bi_anime_dashboard_data.csv')
df_segmented.to_csv(output_file, index=False)

print(f"--- Process Complete! ---")
print(f"Final Consolidated Dataset Shape: {df_segmented.shape[0]} rows, {df_segmented.shape[1]} columns")
print(f"File successfully exported for Power BI to: {output_file}")

--- Step 6: Generating Strategic Metrics per Anime ---
--- Process Complete! ---
Final Consolidated Dataset Shape: 11190 rows, 11 columns
File successfully exported for Power BI to: data/processed\bi_anime_dashboard_data.csv
